# Sesi 8 — Finalize Agentic AI (Orchestrator, Full Qwen Lokal)

Bagian dari kurikulum **Agentic Knowledge System — FastAPI + DuckDB + Qwen (Local, tanpa LLM cloud)**.

> Jalankan sel dari atas ke bawah secara berurutan. Sel bertanda `🔨 TODO` wajib dilengkapi peserta; solusi contoh disediakan pada sel setelahnya.

## 0. Setup Environment

In [ ]:
!pip -q install llama-cpp-python duckdb sentence-transformers fastapi httpx

## 1. Tujuan Sesi Ini
Menggabungkan **Knowledge Agent** (Sesi 1-4) dan **Knowledge ERP** (Sesi 5-7) menjadi satu **orchestrator agent**, seluruhnya di atas satu model **Qwen lokal**, lengkap dengan routing otomatis, evaluasi, dan dokumentasi akhir.

## 1. Model Qwen Lokal
Model yang sama dipakai di seluruh kurikulum (Sesi 3, 6, 4, 7, 8) — cukup diunduh sekali per sesi Colab.

In [ ]:
import os

MODEL_PATH = "qwen2.5-0.5b-instruct-q4_k_m.gguf"
if not os.path.exists(MODEL_PATH):
    !wget -q -O {MODEL_PATH} "https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF/resolve/main/qwen2.5-0.5b-instruct-q4_k_m.gguf"

from llama_cpp import Llama
llm = Llama(model_path=MODEL_PATH, n_ctx=4096, n_threads=4, verbose=False)

def qwen_llm_call(prompt, stop=None):
    out = llm(prompt, max_tokens=300, stop=stop)
    return out["choices"][0]["text"].strip()

## 3. Rekonstruksi Kedua Knowledge Base (Ringkas)

In [ ]:
import duckdb, uuid
from sentence_transformers import SentenceTransformer
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# --- Knowledge Agent ---
kb_con = duckdb.connect("knowledge.duckdb")
kb_con.execute("INSTALL vss; LOAD vss;")
kb_con.execute("""
CREATE TABLE IF NOT EXISTS documents (id VARCHAR PRIMARY KEY, source VARCHAR, content TEXT,
    embedding FLOAT[384], created_at TIMESTAMP DEFAULT current_timestamp);
""")
if kb_con.execute("SELECT COUNT(*) FROM documents").fetchone()[0] == 0:
    for source, text in [("faq_produk.txt", "Garansi produk berlaku 1 tahun sejak tanggal pembelian."),
                         ("sop_retur.txt", "Retur barang maksimal 7 hari setelah barang diterima.")]:
        emb = embed_model.encode(text).tolist()
        kb_con.execute("INSERT INTO documents VALUES (?, ?, ?, ?, current_timestamp)",
                       [str(uuid.uuid4()), source, text, emb])

kb_app = FastAPI()

@kb_app.get("/documents/search")
def search_documents(q: str, k: int = 5):
    q_emb = embed_model.encode(q).tolist()
    rows = kb_con.execute("""
        SELECT id, source, content, array_distance(embedding, ?::FLOAT[384]) AS dist
        FROM documents ORDER BY dist ASC LIMIT ?
    """, [q_emb, k]).fetchall()
    return [{"id": r[0], "source": r[1], "content": r[2], "score": r[3]} for r in rows]

kb_client = TestClient(kb_app)

# --- Knowledge ERP ---
erp_con = duckdb.connect("erp.duckdb")
erp_con.execute("""
CREATE TABLE IF NOT EXISTS products (id VARCHAR PRIMARY KEY, name VARCHAR, price DOUBLE, stock INTEGER);
CREATE TABLE IF NOT EXISTS customers (id VARCHAR PRIMARY KEY, name VARCHAR, email VARCHAR);
CREATE TABLE IF NOT EXISTS orders (id VARCHAR PRIMARY KEY, customer_id VARCHAR, status VARCHAR, created_at TIMESTAMP DEFAULT current_timestamp);
CREATE TABLE IF NOT EXISTS order_items (id VARCHAR PRIMARY KEY, order_id VARCHAR, product_id VARCHAR, qty INTEGER, price DOUBLE);
""")
if erp_con.execute("SELECT COUNT(*) FROM products").fetchone()[0] == 0:
    for name, price, stock in [("Kaos Polos Hitam", 75000, 50), ("Jaket Hoodie", 250000, 15)]:
        erp_con.execute("INSERT INTO products VALUES (?, ?, ?, ?)", [str(uuid.uuid4()), name, price, stock])
if erp_con.execute("SELECT COUNT(*) FROM customers").fetchone()[0] == 0:
    erp_con.execute("INSERT INTO customers VALUES (?, ?, ?)", [str(uuid.uuid4()), "Budi Santoso", "budi@example.com"])

erp_app = FastAPI()

@erp_app.get("/products")
def list_products():
    rows = erp_con.execute("SELECT id, name, price, stock FROM products").fetchall()
    return [{"id": r[0], "name": r[1], "price": r[2], "stock": r[3]} for r in rows]

@erp_app.post("/orders")
def create_order(payload: dict):
    for item in payload["items"]:
        stock = erp_con.execute("SELECT stock FROM products WHERE id=?", [item["product_id"]]).fetchone()
        if not stock or stock[0] < item["qty"]:
            raise HTTPException(400, "Stok tidak cukup")
    order_id = str(uuid.uuid4())
    erp_con.execute("INSERT INTO orders VALUES (?, ?, 'pending', current_timestamp)", [order_id, payload["customer_id"]])
    for item in payload["items"]:
        price = erp_con.execute("SELECT price FROM products WHERE id=?", [item["product_id"]]).fetchone()[0]
        erp_con.execute("INSERT INTO order_items VALUES (?, ?, ?, ?, ?)",
                        [str(uuid.uuid4()), order_id, item["product_id"], item["qty"], price])
        erp_con.execute("UPDATE products SET stock = stock - ? WHERE id=?", [item["qty"], item["product_id"]])
    return {"order_id": order_id, "status": "pending"}

erp_client = TestClient(erp_app)

def execute_tool(name, args):
    if name == "search_knowledge":
        return kb_client.get("/documents/search", params={"q": args["query"], "k": 3}).json()
    if name == "check_stock":
        products = erp_client.get("/products").json()
        return [p for p in products if args["product_name"].lower() in p["name"].lower()]
    return {"error": "unknown tool"}

## 4. Router Berbasis Qwen (Structured Prompting, pola Sesi 4)

In [ ]:
import json, re

def extract_json(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return None

ROUTER_PROMPT = """Klasifikasikan pertanyaan berikut ke satu domain dalam format JSON PERSIS:
{{"domain": "knowledge"}} -> pertanyaan informasi umum/dokumen (FAQ, SOP, kebijakan)
{{"domain": "erp"}} -> pertanyaan data transaksional (stok, harga, order, laporan penjualan)

Pertanyaan: {query}
JSON:"""

def route_query(query: str) -> str:
    raw = qwen_llm_call(ROUTER_PROMPT.format(query=query), stop=["\n\n"])
    parsed = extract_json(raw)
    if parsed and parsed.get("domain") in ("knowledge", "erp"):
        return parsed["domain"]
    erp_keywords = ["stok", "order", "beli", "harga", "pesan", "laporan penjualan"]
    return "erp" if any(kw in query.lower() for kw in erp_keywords) else "knowledge"

## 5. Tool-selection per Domain (pola planner Sesi 4/6) & Orchestrate

In [ ]:
TOOL_PROMPT = """Kamu punya tool berikut untuk domain '{domain}':
{tool_desc}

Tentukan tool dan parameter dalam JSON PERSIS:
{{"tool": "<nama_tool>", "args": {{...}}}}

Pertanyaan: {query}
JSON:"""

TOOL_DESC = {
    "knowledge": '- search_knowledge{"query": str}: mencari dokumen relevan',
    "erp": '- check_stock{"product_name": str}: cek stok & harga produk',
}

def orchestrate(query, verbose=True):
    domain = route_query(query)
    plan_raw = qwen_llm_call(
        TOOL_PROMPT.format(domain=domain, tool_desc=TOOL_DESC[domain], query=query),
        stop=["\n\n"]
    )
    plan = extract_json(plan_raw)
    if verbose:
        print(f"Domain: {domain} | Plan: {plan}")

    context = ""
    tool_used = None
    if plan and "tool" in plan:
        tool_used = plan["tool"]
        result = execute_tool(plan["tool"], plan.get("args", {}))
        context = f"\nHasil: {result}\n"

    final_prompt = f"Jawab dalam Bahasa Indonesia singkat.\n{context}\nPertanyaan: {query}\nJawaban:"
    answer = qwen_llm_call(final_prompt, stop=["\n\n"])
    return {"domain": domain, "tool_used": tool_used, "answer": answer}

print(orchestrate("Berapa lama masa garansi produk?"))
print(orchestrate("Berapa stok Jaket Hoodie yang tersisa?"))

### 🔨 TODO 1
Tambahkan tool `create_order` ke domain `erp` (perluas `TOOL_DESC['erp']` dan `execute_tool`), lengkap dengan konfirmasi manual (`input(...)`) sebelum eksekusi — mengikuti pola guardrail Sesi 6.

In [ ]:
# TODO: tambahkan tool create_order + guardrail konfirmasi


## 6. Evaluasi End-to-End
Uji beberapa skenario campuran (knowledge + ERP), catat domain & tool yang benar-benar dipilih dan waktu respons.

In [ ]:
import pandas as pd, time

eval_questions = [
    "Apa syarat retur barang?",
    "Berapa harga Kaos Polos Hitam?",
    "Berapa lama pengiriman reguler?",
    "____",
]

eval_results = []
for q in eval_questions:
    t0 = time.time()
    result = orchestrate(q, verbose=False)
    eval_results.append({
        "question": q,
        "domain": result["domain"],
        "tool_used": result["tool_used"],
        "answer": result["answer"][:80] + "...",
        "latency_s": round(time.time() - t0, 2)
    })

pd.DataFrame(eval_results)

### 🔨 TODO 2
Lengkapi `eval_questions` menjadi 10 pertanyaan (campuran knowledge & ERP), lalu tinjau manual kolom `domain` dan `tool_used` — apakah router memilih domain/tool yang tepat untuk tiap pertanyaan? Hitung akurasi routing (benar / total).

## 7. Checklist Finalisasi Proyek
- [ ] `docker-compose.yml` menjalankan service Knowledge Agent, Knowledge ERP, dan satu server `llama.cpp` untuk Qwen yang dipakai bersama oleh kedua service.
- [ ] Volume mount agar file `.duckdb` persisten antar restart container.
- [ ] Endpoint `/health` pada tiap service.
- [ ] Frontend sederhana untuk uji manual kedua agent (mirip `frontend/index.html` pada repo referensi).
- [ ] README lengkap: cara run, arsitektur, contoh query.
- [ ] Tabel evaluasi 10 skenario (isian poin 6) disertakan sebagai laporan akhir.

## Ringkasan Sesi 8 — Proyek Selesai 🎉
Orchestrator agent kini bisa merutekan pertanyaan ke Knowledge Agent (RAG) atau Knowledge ERP (aksi transaksional) secara otomatis, seluruhnya berjalan di atas **satu model Qwen lokal** dengan kombinasi dua pola prompting: **ReAct** (Sesi 3, 6) untuk tugas multi-langkah dan **structured JSON planner** (Sesi 4, 7, 8) untuk tugas satu langkah — tanpa dependensi ke LLM cloud sama sekali.

**Langkah lanjut yang disarankan:** deploy ke Docker Compose sesuai struktur folder pada materi utama, coba model Qwen dengan ukuran lebih besar (1.5B/7B) jika kualitas jawaban perlu ditingkatkan, dan perluas evaluasi dengan dataset pertanyaan yang lebih besar.